In [37]:
from __future__ import annotations

import re
import json
import hashlib
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional, Iterable

import fitz  # PyMuPDF
import numpy as np
import pandas as pd
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
import faiss

# -----------------------------
# PATHS (your folder layout)
# -----------------------------
PROJECT_ROOT = Path(".").resolve()
DATA_ROOT = PROJECT_ROOT / "esc_data"  # esc_data/<year>/*.pdf
OUT_DIR = PROJECT_ROOT / "esc_rag_artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Chunking knobs (tune later)
# -----------------------------
SECTION_CHUNK_TARGET_WORDS = 850
SECTION_CHUNK_OVERLAP_WORDS = 120

REC_CHUNK_MIN_WORDS = 35
REC_CHUNK_MAX_WORDS = 240

# -----------------------------
# Embeddings
# -----------------------------
EMBED_MODEL_NAME = "intfloat/e5-large-v2"
EMBED_BATCH_SIZE = 24

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("OUT_DIR:", OUT_DIR)

PROJECT_ROOT: /Users/zak/research/med/RAG_AI/rag2.0
DATA_ROOT: /Users/zak/research/med/RAG_AI/rag2.0/esc_data
OUT_DIR: /Users/zak/research/med/RAG_AI/rag2.0/esc_rag_artifacts


In [38]:
def sha16(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:16]

def normalize_text(s: str) -> str:
    s = s.replace("\u00ad", "")  # soft hyphen
    s = s.replace("￾", "")      # common PDF artifact
    s = s.replace("–", "-").replace("—", "-")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def list_pdf_files(data_root: Path) -> List[Path]:
    if not data_root.exists():
        raise FileNotFoundError(f"Data root not found: {data_root}")
    return sorted(data_root.glob("*/*.pdf"))

def infer_year_from_path(pdf_path: Path) -> Optional[int]:
    # expects esc_data/<year>/file.pdf
    try:
        y = int(pdf_path.parent.name)
        if 1900 <= y <= 2100:
            return y
    except Exception:
        pass
    return None

pdf_files = list_pdf_files(DATA_ROOT)
print(f"Found {len(pdf_files)} PDFs under {DATA_ROOT}")
print("First 5:", [str(p) for p in pdf_files[:5]])

Found 30 PDFs under /Users/zak/research/med/RAG_AI/rag2.0/esc_data
First 5: ['/Users/zak/research/med/RAG_AI/rag2.0/esc_data/2015/pericardial.pdf', '/Users/zak/research/med/RAG_AI/rag2.0/esc_data/2018/myocardial.pdf', '/Users/zak/research/med/RAG_AI/rag2.0/esc_data/2018/myocardial_infraction.pdf', '/Users/zak/research/med/RAG_AI/rag2.0/esc_data/2018/syncope.pdf', '/Users/zak/research/med/RAG_AI/rag2.0/esc_data/2019/dyslipidaemias.pdf']


In [39]:
def extract_pdf_pages(pdf_path: Path) -> List[Dict[str, Any]]:
    doc = fitz.open(pdf_path)
    pages = []
    for i in range(len(doc)):
        page = doc[i]
        text = page.get_text("text")
        pages.append({
            "page_index": i,
            "page_number": i + 1,  # 1-based for citations
            "text_raw": text
        })
    return pages

def get_frequent_lines(pages: List[Dict[str, Any]], min_frac: float = 0.55) -> set:
    """
    Lines that appear on many pages are likely header/footer.
    """
    from collections import Counter
    ctr = Counter()
    for p in pages:
        lines = [normalize_text(l) for l in p["text_raw"].splitlines()]
        lines = [l for l in lines if l and len(l) >= 6]
        for l in set(lines):  # de-dup per page
            ctr[l] += 1
    threshold = int(len(pages) * min_frac)
    return {line for line, c in ctr.items() if c >= threshold}

def remove_frequent_lines(text: str, frequent: set) -> str:
    out_lines = []
    for l in text.splitlines():
        nl = normalize_text(l)
        if nl in frequent:
            continue
        out_lines.append(l)
    return "\n".join(out_lines)

def clean_pages(pages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    frequent = get_frequent_lines(pages, min_frac=0.55)
    out = []
    for p in pages:
        txt = remove_frequent_lines(p["text_raw"], frequent)
        txt = normalize_text(txt)
        out.append({**p, "text_clean": txt})
    return out

In [40]:
def infer_doc_title_and_year(p0_text: str, fallback_year: Optional[int]) -> Tuple[str, Optional[int]]:
    lines = [l.strip() for l in p0_text.splitlines() if l.strip()]
    title = None
    for l in lines[:60]:
        if "Guidelines" in l or "GUIDELINES" in l:
            title = l
            break
    if not title:
        title = lines[0] if lines else "ESC Guideline"

    year = fallback_year
    if year is None:
        m = re.search(r"\b(20\d{2}|19\d{2})\b", p0_text[:2000])
        if m:
            year = int(m.group(1))
    return title, year

# Heading pattern like: "1.2 Title", "3 Recommendations", etc.
HEADING_RE = re.compile(r"^(?P<num>\d+(\.\d+){0,4})\s+(?P<title>[A-Z][^\n]{2,160})$")

def iter_lines_with_page(pages_clean: List[Dict[str, Any]]) -> Iterable[Tuple[int, str]]:
    for p in pages_clean:
        for line in p["text_clean"].splitlines():
            yield p["page_number"], line.strip()

@dataclass
class SectionBlock:
    doc_id: str
    doc_title: str
    year: Optional[int]
    pdf_path: str
    section_num: str
    section_title: str
    page_start: int
    page_end: int
    text: str

def build_sections(
    pages_clean: List[Dict[str, Any]],
    doc_id: str,
    doc_title: str,
    year: Optional[int],
    pdf_path: Path,
) -> List[SectionBlock]:
    current_num = "0"
    current_title = "Front matter"
    buffer_lines: List[str] = []
    page_start = 1
    page_end = 1

    sections: List[SectionBlock] = []

    for page_no, line in iter_lines_with_page(pages_clean):
        if not line:
            continue

        m = HEADING_RE.match(line)
        if m and len(line) <= 180:
            # flush previous
            text = "\n".join(buffer_lines).strip()
            if text:
                sections.append(SectionBlock(
                    doc_id=doc_id,
                    doc_title=doc_title,
                    year=year,
                    pdf_path=str(pdf_path),
                    section_num=current_num,
                    section_title=current_title,
                    page_start=page_start,
                    page_end=page_end,
                    text=text
                ))

            # start new
            current_num = m.group("num")
            current_title = m.group("title").strip()
            buffer_lines = [line]  # keep heading in body
            page_start = page_no
            page_end = page_no
        else:
            buffer_lines.append(line)
            page_end = page_no

    text = "\n".join(buffer_lines).strip()
    if text:
        sections.append(SectionBlock(
            doc_id=doc_id,
            doc_title=doc_title,
            year=year,
            pdf_path=str(pdf_path),
            section_num=current_num,
            section_title=current_title,
            page_start=page_start,
            page_end=page_end,
            text=text
        ))
    return sections

In [41]:
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    doc_title: str
    year: Optional[int]
    pdf_path: str
    chunk_type: str  # "section" or "recommendation"
    section_num: str
    section_title: str
    page_start: int
    page_end: int
    text: str
    signals: Dict[str, Any]

def word_chunks(text: str, target_words: int, overlap_words: int) -> List[str]:
    words = text.split()
    if len(words) <= target_words:
        return [text]
    chunks: List[str] = []
    start = 0
    while start < len(words):
        end = min(len(words), start + target_words)
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        if end == len(words):
            break
        start = max(0, end - overlap_words)
    return chunks

def split_into_paragraphs(text: str) -> List[str]:
    paras = re.split(r"\n\s*\n", text)
    return [p.strip() for p in paras if p.strip()]

REC_SIGNAL_RE = re.compile(
    r"\b(is recommended|are recommended|should be considered|may be considered|is not recommended|are not recommended)\b",
    re.IGNORECASE
)
CLASS_RE = re.compile(r"\bClass\s+(I|IIa|IIb|III)\b", re.IGNORECASE)
LOE_RE = re.compile(r"\b(Level of evidence|LOE)\s*(A|B|C)\b", re.IGNORECASE)

def build_section_chunks(sections: List[SectionBlock]) -> List[Chunk]:
    out: List[Chunk] = []
    for s in sections:
        pieces = word_chunks(
            s.text,
            target_words=SECTION_CHUNK_TARGET_WORDS,
            overlap_words=SECTION_CHUNK_OVERLAP_WORDS
        )
        for i, piece in enumerate(pieces):
            cid = sha16(f"{s.doc_id}|{s.section_num}|section|p{s.page_start}|{i}")
            out.append(Chunk(
                chunk_id=cid,
                doc_id=s.doc_id,
                doc_title=s.doc_title,
                year=s.year,
                pdf_path=s.pdf_path,
                chunk_type="section",
                section_num=s.section_num,
                section_title=s.section_title,
                page_start=s.page_start,
                page_end=s.page_end,
                text=piece,
                signals={"is_recommendation": False}
            ))
    return out

def build_recommendation_chunks(sections: List[SectionBlock]) -> List[Chunk]:
    out: List[Chunk] = []
    for s in sections:
        paras = split_into_paragraphs(s.text)

        rec_paras: List[str] = []
        for p in paras:
            w = p.split()
            if len(w) < REC_CHUNK_MIN_WORDS:
                continue

            if (
                REC_SIGNAL_RE.search(p)
                or CLASS_RE.search(p)
                or LOE_RE.search(p)
                or "Recommendations for" in p
                or "RECOMMENDATIONS" in p
            ):
                rec_paras.append(p)

        for i, rp in enumerate(rec_paras):
            words = rp.split()
            if len(words) > REC_CHUNK_MAX_WORDS:
                rp = " ".join(words[:REC_CHUNK_MAX_WORDS])

            cls = CLASS_RE.search(rp)
            loe = LOE_RE.search(rp)
            signals = {
                "is_recommendation": True,
                "has_class": bool(cls),
                "class": cls.group(1) if cls else None,
                "has_loe": bool(loe),
                "loe": loe.group(2) if loe else None,
            }

            cid = sha16(f"{s.doc_id}|{s.section_num}|rec|p{s.page_start}|{i}")
            out.append(Chunk(
                chunk_id=cid,
                doc_id=s.doc_id,
                doc_title=s.doc_title,
                year=s.year,
                pdf_path=s.pdf_path,
                chunk_type="recommendation",
                section_num=s.section_num,
                section_title=s.section_title,
                page_start=s.page_start,
                page_end=s.page_end,
                text=rp,
                signals=signals
            ))
    return out

In [42]:
pdf_files = list_pdf_files(DATA_ROOT)
print(f"Found {len(pdf_files)} PDFs")

manifest_rows: List[Dict[str, Any]] = []
all_section_chunks: List[Chunk] = []
all_rec_chunks: List[Chunk] = []

for pdf_path in tqdm(pdf_files, desc="Processing PDFs"):
    folder_year = infer_year_from_path(pdf_path)

    try:
        pages = extract_pdf_pages(pdf_path)
        pages_clean = clean_pages(pages)

        title, year = infer_doc_title_and_year(pages_clean[0]["text_clean"], folder_year)
        doc_id = sha16(f"{title}|{year}|{pdf_path.name}")

        sections = build_sections(
            pages_clean=pages_clean,
            doc_id=doc_id,
            doc_title=title,
            year=year,
            pdf_path=pdf_path
        )

        section_chunks = build_section_chunks(sections)
        rec_chunks = build_recommendation_chunks(sections)

        all_section_chunks.extend(section_chunks)
        all_rec_chunks.extend(rec_chunks)

        manifest_rows.append({
            "pdf_path": str(pdf_path),
            "file_name": pdf_path.name,
            "folder_year": folder_year,
            "inferred_year": year,
            "doc_title": title,
            "doc_id": doc_id,
            "pages": len(pages),
            "sections": len(sections),
            "section_chunks": len(section_chunks),
            "rec_chunks": len(rec_chunks),
            "error": None
        })

    except Exception as e:
        manifest_rows.append({
            "pdf_path": str(pdf_path),
            "file_name": pdf_path.name,
            "folder_year": folder_year,
            "error": repr(e)
        })

manifest = pd.DataFrame(manifest_rows)
print("Total docs:", len(manifest))
print("Errors:", manifest["error"].notna().sum())
display(manifest.head(10))

print("Total section chunks:", len(all_section_chunks))
print("Total recommendation chunks:", len(all_rec_chunks))

Found 30 PDFs


Processing PDFs: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [00:12<00:00,  2.35it/s]

Total docs: 30
Errors: 0


,pdf_path,file_name,folder_year,inferred_year,doc_title,doc_id,pages,sections,section_chunks,rec_chunks,error
0,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...,pericardial.pdf,2015,2015,ESC GUIDELINES,7c176406d3aa94c8,44,164,184,35,None
1,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...,myocardial.pdf,2018,2018,2018 ESC/EACTS Guidelines on myocardial,dde2c206310d985b,96,326,399,57,None
2,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...,myocardial_infraction.pdf,2018,2018,Fourth universal definition of myocardial,5d70a1a8b1ff2192,33,105,123,11,None
3,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...,syncope.pdf,2018,2018,2018 ESC Guidelines for the diagnosis and,72918984c295d11d,69,250,286,32,None
4,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...,dyslipidaemias.pdf,2019,2019,2019 ESC/EAS Guidelines for the management,45ed670defb2cd6c,78,314,381,37,None
5,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...,pulmonary_embolism.pdf,2019,2019,2019 ESC Guidelines for the diagnosis and,c7a5899f0c4bd67f,61,209,251,37,None
6,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...,supraventricular_tachycardia.pdf,2019,2019,2019 ESC Guidelines for the management of,00de4fc993786a5a,66,259,303,30,None
7,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...,adult_conginental.pdf,2020,2020,2020 ESC Guidelines for the management of,7185e2f04dc02370,84,361,422,43,None
8,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...,sports_cardiology.pdf,2020,2020,2020 ESC Guidelines on sports cardiology and,072fbf2d3a737a02,80,301,360,59,None
9,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...,cardiac_pacing.pdf,2021,2021,2021 ESC Guidelines on cardiac pacing and,e706777bb449666c,94,241,332,50,None


Total section chunks: 5900
Total recommendation chunks: 571


In [43]:
sec_meta = pd.DataFrame([asdict(c) for c in all_section_chunks])
rec_meta = pd.DataFrame([asdict(c) for c in all_rec_chunks])

manifest.to_parquet(OUT_DIR / "esc_corpus_manifest.parquet", index=False)
sec_meta.to_parquet(OUT_DIR / "esc_sections_meta.parquet", index=False)
rec_meta.to_parquet(OUT_DIR / "esc_recommendations_meta.parquet", index=False)

print("Saved:")
print(" -", OUT_DIR / "esc_corpus_manifest.parquet")
print(" -", OUT_DIR / "esc_sections_meta.parquet")
print(" -", OUT_DIR / "esc_recommendations_meta.parquet")

Saved:
 - /Users/zak/research/med/RAG_AI/rag2.0/esc_rag_artifacts/esc_corpus_manifest.parquet
 - /Users/zak/research/med/RAG_AI/rag2.0/esc_rag_artifacts/esc_sections_meta.parquet
 - /Users/zak/research/med/RAG_AI/rag2.0/esc_rag_artifacts/esc_recommendations_meta.parquet


In [44]:
def embed_passages(model: SentenceTransformer, texts: List[str], batch_size: int = 32) -> np.ndarray:
    passages = [f"passage: {t}" for t in texts]  # E5 format
    emb = model.encode(
        passages,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    return np.asarray(emb, dtype=np.float32)

def build_faiss_index(vectors: np.ndarray) -> faiss.Index:
    dim = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)  # cosine if normalized
    index.add(vectors)
    return index

model = SentenceTransformer(EMBED_MODEL_NAME)

# Sections
sec_texts = sec_meta["text"].tolist()
sec_vecs = embed_passages(model, sec_texts, batch_size=EMBED_BATCH_SIZE)
sec_index = build_faiss_index(sec_vecs)

# Recommendations
rec_texts = rec_meta["text"].tolist()
rec_vecs = embed_passages(model, rec_texts, batch_size=EMBED_BATCH_SIZE)
rec_index = build_faiss_index(rec_vecs)

faiss.write_index(sec_index, str(OUT_DIR / "esc_sections.faiss"))
faiss.write_index(rec_index, str(OUT_DIR / "esc_recommendations.faiss"))

print("Saved indexes:")
print(" -", OUT_DIR / "esc_sections.faiss")
print(" -", OUT_DIR / "esc_recommendations.faiss")

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 7623f790-be93-4055-bee7-8d1443d3be4f)')' thrown while requesting HEAD https://huggingface.co/intfloat/e5-large-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].
Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 24/24 [00:39<00:00,  1.65s/it]

Saved indexes:
 - /Users/zak/research/med/RAG_AI/rag2.0/esc_rag_artifacts/esc_sections.faiss
 - /Users/zak/research/med/RAG_AI/rag2.0/esc_rag_artifacts/esc_recommendations.faiss


In [45]:
def embed_query(model: SentenceTransformer, q: str) -> np.ndarray:
    q_emb = model.encode([f"query: {q}"], normalize_embeddings=True)
    return np.asarray(q_emb, dtype=np.float32)

def search(index: faiss.Index, meta: pd.DataFrame, query: str, k: int = 8) -> pd.DataFrame:
    qv = embed_query(model, query)
    scores, idxs = index.search(qv, k)
    idxs = idxs[0].tolist()
    scores = scores[0].tolist()
    hits = meta.iloc[idxs].copy()
    hits["score"] = scores
    cols = ["score","doc_title","year","section_num","section_title","page_start","chunk_id","pdf_path"]
    return hits[cols].sort_values("score", ascending=False)

test_q = "acute heart failure diuretics recommended class I"
print("Top recommendation hits:")
display(search(rec_index, rec_meta, test_q, k=8))

print("Top section hits:")
display(search(sec_index, sec_meta, test_q, k=8))

Top recommendation hits:


,score,doc_title,year,section_num,section_title,page_start,chunk_id,pdf_path
436,0.885367,2021 ESC Guidelines for the diagnosis and,2021,5.4,Other drugs recommended or to be,24,3661d22f85bb89dd,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...
429,0.858263,2021 ESC Guidelines for the diagnosis and,2021,2.1,What is new,11,6c1143c7b3787813,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...
435,0.858239,2021 ESC Guidelines for the diagnosis and,2021,5.3,Drugs recommended in all patients with heart f...,22,04679067ee289734,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...
487,0.852970,2022 ESC/ERS Guidelines for the diagnosis and,2022,3,"PH, despite a class III recommendation in the ...",8,a63e3d1b3e17e81a,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...
427,0.849466,2021 ESC Guidelines for the diagnosis and,2021,22,References . . . . . . . . . . . . . . . . . ....,4,a4f7c2d7680cabaf,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...
450,0.849406,2021 ESC Guidelines for the diagnosis and,2021,11.2,Clinical presentations,48,ad4f02e2c45bdb81,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...
448,0.847900,2021 ESC Guidelines for the diagnosis and,2021,10.2,Management,41,a08bdaf9cbf7f4dc,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...
58,0.846956,2018 ESC/EACTS Guidelines on myocardial,2018,10.3,Gaps in the evidence,35,18366957376e0f49,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...


Top section hits:


,score,doc_title,year,section_num,section_title,page_start,chunk_id,pdf_path
3533,0.880361,2021 ESC Guidelines for the diagnosis and,2021,5.4,Other drugs recommended or to be,24,9f29b75718722748,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...
4624,0.877895,2023 ESC Guidelines for the management of,2023,2021,ESC Guidelines for the diagnosis and treatment...,46,47939a9f26c8ef45,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...
3686,0.877516,2021 ESC Guidelines for the diagnosis and,2021,2019,ESC Guidelines for the diagnosis and managemen...,97,81533c544ebe7de9,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...
4679,0.875488,2023 ESC Guidelines for the management of,2023,2018,ESC/EACTS guidelines on myocardial revasculari...,84,9a80c661a1c9218a,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...
4649,0.872781,2023 ESC Guidelines for the management of,2023,2017,ESC Guidelines on the diagnosis and treatment ...,63,b4955d9b4fa0533f,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...
3687,0.871227,2021 ESC Guidelines for the diagnosis and,2021,2019,ESC Guidelines for the diagnosis and managemen...,97,42a48cb60ec6dd63,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...
3586,0.870435,2021 ESC Guidelines for the diagnosis and,2021,11.3,Management,51,ac842981f68606ad,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...
3529,0.868702,2021 ESC Guidelines for the diagnosis and,2021,5.2,Pharmacological treatments for,21,9aac3d5d659d222e,/Users/zak/research/med/RAG_AI/rag2.0/esc_data...


In [46]:
from pathlib import Path
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path(".").resolve()
OUT_DIR = PROJECT_ROOT / "esc_rag_artifacts"

sec_index = faiss.read_index(str(OUT_DIR / "esc_sections.faiss"))
rec_index = faiss.read_index(str(OUT_DIR / "esc_recommendations.faiss"))

sec_meta = pd.read_parquet(OUT_DIR / "esc_sections_meta.parquet")
rec_meta = pd.read_parquet(OUT_DIR / "esc_recommendations_meta.parquet")

embed_model = SentenceTransformer("intfloat/e5-large-v2")

print("Loaded:")
print(" - sections vectors:", sec_index.ntotal, "rows meta:", len(sec_meta))
print(" - rec vectors:", rec_index.ntotal, "rows meta:", len(rec_meta))

Loaded:
 - sections vectors: 5900 rows meta: 5900
 - rec vectors: 571 rows meta: 571


In [47]:
def embed_query(model: SentenceTransformer, q: str) -> np.ndarray:
    v = model.encode([f"query: {q}"], normalize_embeddings=True)
    return np.asarray(v, dtype=np.float32)

def faiss_search(index: faiss.Index, meta: pd.DataFrame, q: str, k: int) -> pd.DataFrame:
    qv = embed_query(embed_model, q)
    scores, idxs = index.search(qv, k)
    idxs = idxs[0].tolist()
    scores = scores[0].tolist()
    hits = meta.iloc[idxs].copy()
    hits["faiss_score"] = scores
    return hits

def retrieve_candidates(query: str, k_sections: int = 25, k_recs: int = 25) -> pd.DataFrame:
    a = faiss_search(sec_index, sec_meta, query, k_sections)
    b = faiss_search(rec_index, rec_meta, query, k_recs)
    a["source_index"] = "sections"
    b["source_index"] = "recommendations"
    candidates = pd.concat([a, b], ignore_index=True)

    # Deduplicate by chunk_id (rare but can happen)
    candidates = candidates.sort_values("faiss_score", ascending=False).drop_duplicates("chunk_id")

    return candidates.reset_index(drop=True)

# quick test
cands = retrieve_candidates("acute heart failure admission criteria oxygen saturation", 20, 20)
cands[["faiss_score","source_index","doc_title","year","section_num","section_title","page_start","chunk_id"]].head(8)

,faiss_score,source_index,doc_title,year,section_num,section_title,page_start,chunk_id
0,0.865629,sections,2021 ESC Guidelines for the diagnosis and,2021,2019,ESC Guidelines for the diagnosis and managemen...,97,1da3729ef132d7bc
1,0.862186,sections,2025 ESC Guidelines for the management,2025,2021,ESC Guidelines for the diagnosis and treatment...,74,6a772c872696d8f9
2,0.862178,sections,2021 ESC Guidelines for the diagnosis and,2021,11.2,Clinical presentations,48,0b5c3c3ea6d8a18c
3,0.859601,sections,2021 ESC Guidelines for the diagnosis and,2021,11.2.1,Acute decompensated heart failure . . . . . . ...,3,81ae3ff7130e00ee
4,0.859100,sections,2021 ESC Guidelines for the diagnosis and,2021,11,Acute heart failure,46,2ba8048e76525013
5,0.858419,recommendations,2021 ESC Guidelines for the diagnosis and,2021,11.2,Clinical presentations,48,ad4f02e2c45bdb81
6,0.856700,sections,2021 ESC Guidelines for the diagnosis and,2021,11.3,Management,51,ac842981f68606ad
7,0.856568,sections,Guidelines for the diagnosis and treatment,2023,2023,Focused Update of the 2021 ESC,1,87468aaf48629c8f


In [48]:
# If needed:
# !pip install -q sentence-transformers

from sentence_transformers import CrossEncoder

RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANK_MODEL_NAME)

print("Reranker loaded:", RERANK_MODEL_NAME)

Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [49]:
def rerank(query: str, candidates: pd.DataFrame, top_k: int = 10) -> pd.DataFrame:
    if len(candidates) == 0:
        return candidates

    pairs = [(query, t) for t in candidates["text"].tolist()]
    scores = reranker.predict(pairs)  # higher = better
    out = candidates.copy()
    out["rerank_score"] = scores

    out = out.sort_values("rerank_score", ascending=False).head(top_k).reset_index(drop=True)
    return out

reranked = rerank("acute heart failure admission criteria oxygen saturation", cands, top_k=10)
reranked[["rerank_score","source_index","doc_title","year","section_num","section_title","page_start","chunk_id"]]

,rerank_score,source_index,doc_title,year,section_num,section_title,page_start,chunk_id
0,0.351527,recommendations,2021 ESC Guidelines for the diagnosis and,2021,11.2,Clinical presentations,48,ad4f02e2c45bdb81
1,0.025041,sections,2021 ESC Guidelines for the diagnosis and,2021,11.2,Clinical presentations,48,0b5c3c3ea6d8a18c
2,-1.631561,sections,2021 ESC Guidelines for the diagnosis and,2021,11.3,Management,51,ac842981f68606ad
3,-1.792529,sections,2021 ESC Guidelines for the diagnosis and,2021,2019,ESC Guidelines for the diagnosis and managemen...,97,1da3729ef132d7bc
4,-2.411865,sections,2021 ESC Guidelines for the diagnosis and,2021,11.3,Management,51,a1eb0598dee7a2f0
5,-2.615119,sections,2022 ESC/ERS Guidelines for the diagnosis and,2022,3,PH). Non-matched perfusion defects similar to ...,26,d48eeda5fa00b413
6,-2.653623,recommendations,2021 ESC Guidelines for the diagnosis and,2021,11.3,Management,51,73722a8a0aae7f31
7,-2.902680,sections,2019 ESC Guidelines for the diagnosis and,2019,17,References,49,d76e4c1ae2742f03
8,-2.988530,sections,2023 ESC Guidelines for the management of,2023,2021,ESC Guidelines for the diagnosis and treatment...,46,3832130b70d65863
9,-3.111073,sections,2021 ESC Guidelines for the diagnosis and,2021,2019,ESC Guidelines for the diagnosis and managemen...,97,81533c544ebe7de9


In [50]:
def format_citation_row(row: pd.Series) -> str:
    # Example: "ESC HF Guidelines (2021), p.123, §5.2 Diuretics, chunk=abc123"
    title = str(row.get("doc_title", "")).strip()
    year = row.get("year", None)
    secnum = str(row.get("section_num", "")).strip()
    sectitle = str(row.get("section_title", "")).strip()
    p = int(row.get("page_start", 0))
    chunk_id = str(row.get("chunk_id", ""))

    # If your recommendation signals include class/loe, show them
    signals = row.get("signals", {})
    if isinstance(signals, dict) and signals.get("is_recommendation"):
        cls = signals.get("class")
        loe = signals.get("loe")
        extra = []
        if cls: extra.append(f"Class {cls}")
        if loe: extra.append(f"LOE {loe}")
        extra_str = f" ({', '.join(extra)})" if extra else ""
    else:
        extra_str = ""

    yr_str = f"({int(year)})" if year is not None and str(year).isdigit() else ""
    sec_str = f"§{secnum} {sectitle}".strip()
    return f"{title} {yr_str}, p.{p}, {sec_str}{extra_str}, chunk={chunk_id}"

def citation_object(row: pd.Series) -> dict:
    signals = row.get("signals", {})
    if not isinstance(signals, dict):
        signals = {}
    return {
        "chunk_id": row.get("chunk_id"),
        "doc_title": row.get("doc_title"),
        "year": row.get("year"),
        "page": int(row.get("page_start", 0)),
        "section_num": row.get("section_num"),
        "section_title": row.get("section_title"),
        "chunk_type": row.get("chunk_type"),
        "class": signals.get("class"),
        "loe": signals.get("loe"),
        "pdf_path": row.get("pdf_path"),
    }

# Demo
for i in range(min(3, len(reranked))):
    print(format_citation_row(reranked.iloc[i]))

2021 ESC Guidelines for the diagnosis and (2021), p.48, §11.2 Clinical presentations, chunk=ad4f02e2c45bdb81
2021 ESC Guidelines for the diagnosis and (2021), p.48, §11.2 Clinical presentations, chunk=0b5c3c3ea6d8a18c
2021 ESC Guidelines for the diagnosis and (2021), p.51, §11.3 Management, chunk=ac842981f68606ad


In [51]:
def build_context_bundle(query: str, top_k_final: int = 10) -> dict:
    candidates = retrieve_candidates(query, k_sections=30, k_recs=40)
    top = rerank(query, candidates, top_k=top_k_final)

    citations = [citation_object(top.iloc[i]) for i in range(len(top))]
    citations_pretty = [format_citation_row(top.iloc[i]) for i in range(len(top))]

    # You can choose to pass only text or text + inline markers
    context_blocks = []
    for i in range(len(top)):
        cid = top.iloc[i]["chunk_id"]
        txt = top.iloc[i]["text"]
        context_blocks.append(f"[{cid}]\n{txt}")

    return {
        "query": query,
        "context_text": "\n\n---\n\n".join(context_blocks),
        "citations": citations,
        "citations_pretty": citations_pretty,
        "top_table": top
    }

bundle = build_context_bundle("acute heart failure initial management diuretics oxygen saturation", top_k_final=8)
print("\n".join(bundle["citations_pretty"][:3]))

2021 ESC Guidelines for the diagnosis and (2021), p.51, §11.3 Management, chunk=ac842981f68606ad
2021 ESC Guidelines for the diagnosis and (2021), p.51, §11.3 Management, chunk=a1eb0598dee7a2f0
2021 ESC Guidelines for the diagnosis and (2021), p.51, §11.3 Management, chunk=15b272d975dcd452


In [52]:
def available_citation_ids(bundle: dict) -> set:
    return {c["chunk_id"] for c in bundle["citations"]}

print("Allowed citation ids:", list(available_citation_ids(bundle))[:5])

Allowed citation ids: ['81533c544ebe7de9', '80818fc59251d357', '42a48cb60ec6dd63', '3453e92fba0152c9', '15b272d975dcd452']


In [73]:
from openai import OpenAI
import json

client = OpenAI()
MODEL_NAME = "gpt-5.2"

print("OpenAI ready")

OpenAI ready


In [88]:
def call_gpt_json(system_prompt: str, user_prompt: str):
    response = client.responses.create(
        model=MODEL_NAME,
        temperature=0.0,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
    )

    output_text = response.output_text.strip()

    try:
        return json.loads(output_text)
    except Exception:
        print("Model did not return valid JSON.")
        print("Raw output:\n", output_text)
        raise

In [113]:
SYSTEM_PROMPT_RAG = """
You are a clinical decision-support assistant for research purposes only.

You must:
- Use ONLY the provided guideline context.
- Cite chunk IDs in square brackets after every factual claim.
- Never invent guideline statements.
- If unsupported, write:
  "Cannot be determined from provided data and retrieved guidelines."

CRITICAL:
- Output strictly valid JSON.
- No markdown.
- No explanations.
- Single JSON object only.
"""

SYSTEM_PROMPT_RAG += """

Consistency rule:
- Do not list a diagnosis as likely if it is contradicted by provided findings (e.g., SVT when ECG shows sinus rhythm at 75 bpm).
"""
SYSTEM_PROMPT_RAG += """
Ranking rule (critical):
- Rank diagnoses by overall fit to the provided case (history + CXR + ECG + labs + ABG), not by availability of guideline text.
- Do not include chronic entities (e.g., CTEPH) in the top 3 unless the case explicitly includes chronic time-course features (months), prior PE, or prior imaging supporting it.
"""
SYSTEM_PROMPT_RAG += """
Safety note:
- If the case contains a clearly critical abnormality (e.g., potassium ≥6.0 mmol/L), include a test/action to address it even if guideline context is sparse; if no ESC citation exists, label it explicitly as 'Clinical safety item (non-ESC)'.
"""

In [128]:
task_instructions_A = """
Return STRICT JSON with EXACT keys (no extra keys). Keep it SHORT.

{
  "diagnoses_ranked": [
    {"diagnosis": "string", "why": "≤1 sentence, must include [chunk_id]"}
  ],
  "tests_ranked": [
    {"test": "string", "timing": "Day 0 ER | Day 1 inpatient | Discharge/outpatient", "why": "≤1 sentence with [chunk_id]"}
  ],
  "disposition": {
    "answer": "DISCHARGE | ADMIT | Cannot be determined from provided data and retrieved guidelines.",
    "why": "≤2 sentences with [chunk_id]"
  },
  "admit_to": {
    "answer": "CARDIOLOGY DEPARTMENT | CICU | RESPIRATORY DEPARTMENT (PULMONOLOGISTS) | INTERNAL MEDICINE | ICU | SURGICAL | Cannot be determined from provided data and retrieved guidelines.",
    "why": "≤2 sentences with [chunk_id]"
  },
  "prognosis_per_diagnosis": [
    {"diagnosis": "string", "score_1_to_8": 1, "why": "≤1 sentence with [chunk_id] OR Cannot be determined..."}
  ]
}

Hard limits:
- diagnoses_ranked: exactly 3 items
- tests_ranked: max 6 items
- prognosis_per_diagnosis: exactly 3 items

Citations:
- Every 'why' must include at least one [chunk_id], unless the value is exactly the Cannot-be-determined sentence.
No markdown.
"""
task_instructions_A += """
Coverage rule:
- At least 1 diagnosis must be cardiovascular/HF-related if the case history includes HFrEF/CAD AND CXR suggests congestion/cardiomegaly.
- Do not propose V/Q scan for CTEPH unless the case explicitly includes persistent dyspnoea after PE or ≥3 months time course.
"""
task_instructions_A += """
Constraint:
- The 3 diagnoses must be DISTINCT disease processes. Do not list the same diagnosis twice with different risk categories (e.g., PE non-high-risk vs intermediate-risk).
"""
task_instructions_A = task_instructions_A.replace(
    '"admit_to": {',
    '"admit_to": {\n    "answer_rule": "If ADMIT, choose the most appropriate department based on the case + retrieved context; if context is insufficient, answer INTERNAL MEDICINE (default ward) and state why.",'
)
task_instructions_A += """
Additional constraints:
- Diagnosis #3 must be a DISTINCT acute alternative diagnosis or acute trigger relevant to the presentation (not a chronic phenotype) unless the case explicitly suggests chronic progression over months.
- If you include a "Clinical safety item (non-ESC)" then its why must be exactly: "Clinical safety item (non-ESC)."
- If PE appears in diagnoses_ranked, tests_ranked must include ONE diagnostic step for PE evaluation (e.g., clinical probability + imaging), supported by [chunk_id] if available.
"""

In [121]:
bundle = build_context_bundle(case_text, top_k_final=12)

user_prompt = build_user_prompt(
    case_text,
    bundle,
    task_instructions
)

response_json = call_gpt_json(
    SYSTEM_PROMPT_RAG,
    user_prompt
)

response_json

{'diagnoses_ranked': [{'diagnosis': 'Acute decompensated heart failure (HFrEF) with pulmonary congestion',
   'rationale': 'Dyspnoea is a typical symptom of heart failure. [39f98d255431b22e] Chest radiography is a recommended diagnostic test in suspected chronic heart failure, and the provided CXR shows increased cardiothoracic ratio with prominent interstitial markings bilaterally, which fits pulmonary congestion in a patient with known HFrEF. [39f98d255431b22e]'},
  {'diagnosis': 'Acute pulmonary embolism (non-high-risk suspected)',
   'rationale': 'D-dimer is listed among tests with diagnostic utility in the diagnostic work-up to find the reason for dyspnoea, and the case has an elevated D-dimer. [5574c0be7a1aa38e] The patient is hemodynamically stable (normotensive), which is compatible with acute PE without haemodynamic compromise. [331c9fd8d3d0cbaa]'},
  {'diagnosis': 'Supraventricular tachycardia (SVT) causing dyspnoea',
   'rationale': 'Cannot be determined from provided data a

In [122]:
case_text = """Info:
Age:
Male
 
Signs and symptoms:
Dyspnea on exertion since 5 days – Dyspnea started 4 hours before
 
T= 36o C
Blood pressure= 166/110mmHg
Hemodynamically stable: Yes
Tachypneic
GCS= 15/15
 
Laboratory exams:
• WBC = 9.30 Κ/μl (normal: 4.6–10.20 Κ/μl)
• Neutrophils = 5.53 Κ/μl (normal: 1.79–7.50 Κ/μl)
• HGB = 17 g/dl (normal: 12.2–18.1 g/dl)
• HCT = 54 % (normal: 37.7–53.7 %) ↑
• PLT = 211 Κ/μl (normal: 140.0–450.0 Κ/μl)
• BNP = — (not provided)
• TnI = 22 pg/ml (normal: <34 pg/ml)
• CRP = 1 mg/dl (normal: <0.6 mg/dl) ↑
• Urea = 57 mg/dl (normal: 15–45 mg/dl) ↑
• Creatinine = 1 mg/dl (normal: 0.7–1.2 mg/dl)
• Glucose = 100 mg/dl (normal: 70–115 mg/dl)
• Total bilirubin = 1.1 mg/dl (normal: 0.3–1.2 mg/dl)
• SGOT (AST) = 48 IU/L (normal: 5–40 IU/L) ↑
• SGPT (ALT) = 27 IU/L (normal: 5–35 IU/L)
• Γ-GT = 27 IU/L (normal: 10–49 IU/L)
• ALP = 86 IU/L (normal: 25–125 IU/L)
• Sodium = 137 mmol/L (normal: 136–145 mmol/L)
• Potassium = 6.1 mmol/L (normal: 3.5–5.1 mmol/L) ↑
• APTT = 55 sec (normal: 28.0–40.0 sec) ↑
• PT = 25 sec (normal: 12.0–15.0 sec) ↑
• D-Dimers = 0.87 μg/ml (normal: <0.5 μg/ml) ↑
 
🩸 Arterial Blood Gases (FiO₂ 21%)
• pH = 7.38
• pO₂ = 65 mmHg
• pCO₂ = 33 mmHg
• Lactate = 1.1 mmol/L
• SpO₂ = 92 %
• FiO₂ = 21 %
 
 
ECG:
Sinus Rhythm- BPM= 75/min –LBBB
 
CXR:
Increased cardiothoracic ratio - Prominent interstitial markings (BOTH SIDES)
 
Other exams:
 
Patients’ history:
Heart failure with reduced ejection fraction - Coronary disease – NSTEMI Myocardial infarction – Coronary artery bypass graft – Percutaneous coronary intervention after CABG due to NSTEMI – Dyslipidemia
 
Questions for the AI (before the diagnosis):
1. Please list three potential diagnoses, in order of likelihood (from most to least probable).
2. Which further tests should be requested, and when should they be performed (Which day?, in the ER department or not?, after hospitalization or discharge?). Answer from most important to less important.
3. Should I discharge the patient?
4. If you suggest admission, in which department should the patient be admitted? (CARDIOLOGY DEPARTMENT, CICU, RESPIRATORY DEPARTMENT (PULMONOLOGISTS), INTERNAL MEDICINE, ICU, SURGICAL)
5. What is the prognosis for each diagnosis? Answer for each in a scale (one value, not a range):
Score
Label
Clinical Implication
1
Safe for Discharge from ER
Fully stable, no further inpatient management needed.
2
Short Observation (<24h)
Requires brief monitoring or diagnostic workup; likely to go home next day.
3
Short Admission (1–3 days)
Needs treatment/monitoring; can be discharged within 72h.
4
Standard Admission (3–7 days)
Requires therapy, imaging, and gradual stabilization; moderate severity.
5
Prolonged Admission (>7 days)
Complex management, likely complications, slow response to therapy.
6
High Risk of ICU
Hemodynamic instability; ICU likely.
7
High Risk of Intubation
Hemodynamic or respiratory instability; ICU likely.
8
High Risk of Death (In-hospital)
Critical illness, multiorgan risk, or very poor cardiac/respiratory reserve.
 """

task_instructions = """
Answer questions 1–5 exactly in the required JSON structure:

{
  "diagnoses_ranked": [...],
  "tests_and_timing_ranked": [...],
  "disposition": {...},
  "admit_to": {...},
  "prognosis_per_diagnosis": [...]
}

Prognosis must be integer 1–8.
Department must be from:
["CARDIOLOGY DEPARTMENT","CICU","RESPIRATORY DEPARTMENT (PULMONOLOGISTS)","INTERNAL MEDICINE","ICU","SURGICAL"]
"""

bundle = build_context_bundle(case_text, top_k_final=12)
user_prompt = build_user_prompt(case_text, bundle, task_instructions)

response_json = call_gpt_json(
    SYSTEM_PROMPT_RAG,
    user_prompt
)

response_json

{'diagnoses_ranked': [{'diagnosis': 'Acute decompensated heart failure (HFrEF) with pulmonary congestion',
   'rationale': 'Dyspnoea is a typical symptom of heart failure. [39f98d255431b22e] Chest radiography is a recommended test in suspected chronic heart failure and can show pulmonary congestion; this case CXR reports increased cardiothoracic ratio and prominent bilateral interstitial markings, supporting congestion. [39f98d255431b22e]'},
  {'diagnosis': 'Acute pulmonary embolism (non-high-risk suspected)',
   'rationale': 'Dyspnoea work-up may include D-dimer and arterial blood gases, both of which have diagnostic utility but limited specificity; this case has elevated D-dimer and hypoxaemia on ABG. [5574c0be7a1aa38e] PE risk assessment begins upon suspicion and includes oxygen saturation; this case has SpO2 92% (not <90%). [d0eb475a0d310c15]'},
  {'diagnosis': 'Supraventricular tachycardia (SVT) causing dyspnoea',
   'rationale': 'SVT can present with dyspnoea. [657d229e4be0f22f] 

In [93]:
from textwrap import indent

PROGNOSIS_SCALE = {
    1: "Safe for Discharge from ER",
    2: "Short Observation (<24h)",
    3: "Short Admission (1–3 days)",
    4: "Standard Admission (3–7 days)",
    5: "Prolonged Admission (>7 days)",
    6: "High Risk of ICU",
    7: "High Risk of Intubation",
    8: "High Risk of Death (In-hospital)"
}

def render_readable_report(resp: dict) -> str:
    lines = []
    lines.append("=== RAG Output (Readable View) ===\n")

    # 1) Diagnoses
    lines.append("1) Differential diagnoses (ranked)")
    dxs = resp.get("diagnoses_ranked", [])
    if isinstance(dxs, list) and dxs:
        for i, d in enumerate(dxs, 1):
            if isinstance(d, dict):
                dx = d.get("diagnosis", "")
                rat = d.get("rationale", "")
                lines.append(f"  {i}. {dx}")
                if rat:
                    lines.append(indent(f"Reason: {rat}", "     "))
            else:
                lines.append(f"  {i}. {str(d)}")
    else:
        lines.append("  (none)")
    lines.append("")

    # 2) Tests
    lines.append("2) Further tests (ranked) + timing")
    tests = resp.get("tests_and_timing_ranked", [])
    if isinstance(tests, list) and tests:
        for i, t in enumerate(tests, 1):
            if isinstance(t, dict):
                test = t.get("test", "")
                timing = t.get("timing", "")
                lines.append(f"  {i}. {test}")
                if timing:
                    lines.append(indent(f"Timing: {timing}", "     "))
            else:
                lines.append(f"  {i}. {str(t)}")
    else:
        lines.append("  (none)")
    lines.append("")

    # 3) Disposition
    lines.append("3) Disposition")
    disp = resp.get("disposition", resp.get("disposition_decision", {}))
    if isinstance(disp, dict):
        sd = disp.get("should_discharge", disp.get("decision", ""))
        rat = disp.get("rationale", "")
        lines.append(f"  Should discharge? {sd}")
        if rat:
            lines.append(indent(f"Reason: {rat}", "  "))
    else:
        lines.append(f"  {disp}")
    lines.append("")

    # 4) Admission location
    lines.append("4) Admission destination")
    adm = resp.get("admit_to", {})
    if isinstance(adm, dict):
        dep = adm.get("department", "")
        rat = adm.get("rationale", "")
        lines.append(f"  Department: {dep}")
        if rat:
            lines.append(indent(f"Reason: {rat}", "  "))
    else:
        lines.append(f"  {adm}")
    lines.append("")

    # 5) Prognosis table
    lines.append("5) Prognosis per diagnosis (score 1–8)")
    prog = resp.get("prognosis_per_diagnosis", [])
    if isinstance(prog, list) and prog:
        for p in prog:
            if isinstance(p, dict):
                dx = p.get("diagnosis", "")
                sc = p.get("prognosis_score_1_to_8", p.get("prognosis_score", None))
                rat = p.get("rationale", "")
                label = PROGNOSIS_SCALE.get(sc, "Unknown") if isinstance(sc, int) else "Unknown"
                lines.append(f"  - {dx}: {sc} — {label}")
                if rat:
                    lines.append(indent(f"Reason: {rat}", "    "))
            else:
                lines.append(f"  - {str(p)}")
    else:
        lines.append("  (none)")

    return "\n".join(lines)

In [94]:
print(render_readable_report(response_json))

# raw JSON still available for your evaluation pipeline:
response_json

=== RAG Output (Readable View) ===

1) Differential diagnoses (ranked)
  1. Heart failure (chronic HFrEF with possible decompensation presenting with breathlessness/dyspnoea)
     Reason: Breathlessness is a typical symptom of heart failure. [39f98d255431b22e] Chest radiography is a recommended diagnostic test in suspected chronic heart failure, and the provided CXR abnormalities are being used here only as supportive context rather than a guideline-defined criterion. [39f98d255431b22e]
  2. Acute pulmonary embolism (risk stratification needed; currently normotensive/hemodynamically stable)
     Reason: D-dimer has diagnostic utility in the work-up to find the reason for dyspnoea (limited specificity). [5574c0be7a1aa38e] Arterial blood gases have diagnostic utility in the work-up to find the reason for dyspnoea (limited specificity). [5574c0be7a1aa38e] PE severity/risk assessment uses parameters including male sex, chronic heart failure, and arterial oxyhaemoglobin saturation <90%. [d0

{'diagnoses_ranked': [{'diagnosis': 'Heart failure (chronic HFrEF with possible decompensation presenting with breathlessness/dyspnoea)',
   'rationale': 'Breathlessness is a typical symptom of heart failure. [39f98d255431b22e] Chest radiography is a recommended diagnostic test in suspected chronic heart failure, and the provided CXR abnormalities are being used here only as supportive context rather than a guideline-defined criterion. [39f98d255431b22e]'},
  {'diagnosis': 'Acute pulmonary embolism (risk stratification needed; currently normotensive/hemodynamically stable)',
   'rationale': 'D-dimer has diagnostic utility in the work-up to find the reason for dyspnoea (limited specificity). [5574c0be7a1aa38e] Arterial blood gases have diagnostic utility in the work-up to find the reason for dyspnoea (limited specificity). [5574c0be7a1aa38e] PE severity/risk assessment uses parameters including male sex, chronic heart failure, and arterial oxyhaemoglobin saturation <90%. [d0eb475a0d310c

In [95]:
case_text_post = f"""
{case_text}

NEW INFORMATION (after diagnosis):
The diagnosis was mainly acute HFrEF. (There may be additional diagnoses that I will not provide—please take this into account in your workup.)

Bedside ECHO:
- LVEF = 35%
- Dilated LV
- RV normal size and contractility
- mild Mitral regurgitation
- mild Tricuspid regurgitation
- mild Aortic valve stenosis
"""

In [105]:
task_instructions_B = """
Return STRICT JSON with EXACT keys (no extra keys). Keep it SHORT.

{
  "q6_prognosis_score_1_to_8": 1-8,
  "q7_trigger": {
    "answer": "string OR Cannot be determined from provided data and retrieved guidelines.",
    "evidence": ["max 3 bullets, each ≤1 line and includes [chunk_id]"]
  },
  "q8_admit_to": {
    "answer": "CICU | CARDIOLOGY DEPARTMENT | Cannot be determined from provided data and retrieved guidelines.",
    "why": "≤2 sentences with [chunk_id]"
  },
  "q9_cause_workup": [
    {"step": 1, "do": "string", "when": "Day 0 ER | Day 1 inpatient | Discharge/outpatient", "cite": ["chunk_id"]}
  ],
  "q10_treatment_plan": [
    {"day": "Day 0", "do": ["max 4 bullets"], "cite": ["chunk_id"]},
    {"day": "Day 1", "do": ["max 4 bullets"], "cite": ["chunk_id"]},
    {"day": "Discharge", "do": ["max 4 bullets"], "cite": ["chunk_id"]}
  ],
  "q11_followup": {
    "in_hospital": ["max 5 bullets with [chunk_id]"],
    "after_discharge": ["max 5 bullets with [chunk_id] OR Cannot be determined..."]
  },
  "q12_hds_score_1_to_5": 1-5
}

Rules:
- No markdown.
- No long paragraphs.
- Every bullet must include at least one [chunk_id] unless it is exactly the Cannot-be-determined sentence.
"""

In [99]:
bundle_post = build_context_bundle(case_text_post, top_k_final=14)

user_prompt_post = build_user_prompt(
    case_text_post,
    bundle_post,
    task_instructions_post
)

response_json_post = call_gpt_json(
    SYSTEM_PROMPT_RAG,
    user_prompt_post
)

response_json_post

{'q6_prognosis_score_1_to_8': 4,
 'q7_trigger_of_decompensation': {'most_likely': 'Cannot be determined from provided data and retrieved guidelines.',
  'supporting_points': ['Acute decompensated heart failure is a common presentation of acute heart failure, but specific triggers are not provided in the retrieved context. [ad4f02e2c45bdb81]']},
 'q8_admission_location': {'choice': 'CARDIOLOGY DEPARTMENT',
  'rationale': 'Acute heart failure evaluation includes ECG, chest X-ray, echocardiography, natriuretic peptides, troponin, creatinine, and electrolytes measured on admission and during hospitalization, supporting inpatient cardiology management. [ad4f02e2c45bdb81]'},
 'q9_cause_workup_algorithm': [{'step': 1,
   'do': 'Measure natriuretic peptides (BNP/NT-proBNP).',
   'where': 'ER',
   'day': 'Day 0',
   'why': 'Natriuretic peptides are recommended in acute heart failure on admission and have high negative predictive value. [ad4f02e2c45bdb81]'},
  {'step': 2,
   'do': 'Perform trans

In [101]:
HDS_SCALE = {
    1: "Discharged from ER (<24h)",
    2: "Short Stay (1–2 days)",
    3: "Standard Stay (3–5 days)",
    4: "Prolonged Stay (6–10 days)",
    5: "Extended Stay (>10 days)"
}

def render_query_b_report(resp: dict) -> str:
    lines = []
    lines.append("=== RAG Output (Post-diagnosis) — Readable View ===\n")

    # Q6
    q6 = resp.get("q6_prognosis_score_1_to_8", None)
    lines.append(f"6) Prognosis score (1–8): {q6}")
    lines.append("")

    # Q7
    q7 = resp.get("q7_trigger_of_decompensation", {})
    lines.append("7) Most likely trigger of decompensation")
    if isinstance(q7, dict):
        lines.append(f"  - Most likely: {q7.get('most_likely','')}")
        pts = q7.get("supporting_points", [])
        if pts:
            lines.append("  - Supporting points:")
            for p in pts:
                lines.append(f"     • {p}")
    lines.append("")

    # Q8
    q8 = resp.get("q8_admission_location", {})
    lines.append("8) Admit to")
    if isinstance(q8, dict):
        lines.append(f"  - Choice: {q8.get('choice','')}")
        lines.append(f"  - Rationale: {q8.get('rationale','')}")
    lines.append("")

    # Q9
    q9 = resp.get("q9_cause_workup_algorithm", [])
    lines.append("9) Workup to identify the cause (algorithm)")
    if isinstance(q9, list) and q9:
        for step in q9:
            lines.append(f"  Step {step.get('step')}: {step.get('do')}")
            lines.append(f"     Where: {step.get('where')} | When: {step.get('day')}")
            lines.append(f"     Why: {step.get('why')}")
    else:
        lines.append("  (none)")
    lines.append("")

    # Q10
    q10 = resp.get("q10_treatment_plan_by_day", [])
    lines.append("10) Treatment plan by day")
    if isinstance(q10, list) and q10:
        for d in q10:
            lines.append(f"  {d.get('day')}:")
            for a in d.get("actions", []):
                lines.append(f"     • {a}")
            cits = d.get("citations", [])
            if cits:
                lines.append(f"     Citations: {', '.join(cits)}")
    else:
        lines.append("  (none)")
    lines.append("")

    # Q11
    q11 = resp.get("q11_followup_during_hospitalization_and_after_discharge", [])
    lines.append("11) Follow-up investigations / interventions")
    if isinstance(q11, list) and q11:
        for phase in q11:
            lines.append(f"  {phase.get('phase')}:")
            for item in phase.get("items_ranked", []):
                lines.append(f"     • {item}")
    else:
        lines.append("  (none)")
    lines.append("")

    # Q12
    q12 = resp.get("q12_estimated_duration_hds_1_to_5", None)
    label = HDS_SCALE.get(q12, "Unknown") if isinstance(q12, int) else "Unknown"
    lines.append(f"12) Estimated hospitalization duration (HDS-5): {q12} — {label}")

    return "\n".join(lines)

In [102]:
print(render_query_b_report(response_json_post))

=== RAG Output (Post-diagnosis) — Readable View ===

6) Prognosis score (1–8): 4

7) Most likely trigger of decompensation
  - Most likely: Cannot be determined from provided data and retrieved guidelines.
  - Supporting points:
     • Acute decompensated heart failure is a common presentation of acute heart failure, but specific triggers are not provided in the retrieved context. [ad4f02e2c45bdb81]

8) Admit to
  - Choice: CARDIOLOGY DEPARTMENT
  - Rationale: Acute heart failure evaluation includes ECG, chest X-ray, echocardiography, natriuretic peptides, troponin, creatinine, and electrolytes measured on admission and during hospitalization, supporting inpatient cardiology management. [ad4f02e2c45bdb81]

9) Workup to identify the cause (algorithm)
  Step 1: Measure natriuretic peptides (BNP/NT-proBNP).
     Where: ER | When: Day 0
     Why: Natriuretic peptides are recommended in acute heart failure on admission and have high negative predictive value. [ad4f02e2c45bdb81]
  Step 2: Pe

In [129]:
def call_gpt_text(system_prompt: str, user_prompt: str) -> str:
    resp = client.responses.create(
        model=MODEL_NAME,
        temperature=0.0,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return resp.output_text.strip()

QUERYGEN_SYSTEM = "You generate search queries for retrieving guideline passages. Output only JSON."

def generate_retrieval_queries_llm(case_text: str, mode: str) -> list[str]:
    # mode: "A" (pre-dx) or "B" (post-dx)
    prompt = f"""
CASE:
{case_text}

MODE: {mode}

Task:
Generate 8 to 12 short retrieval queries to find the most relevant ESC guideline passages.
Focus on: admission vs discharge, triage level (ward vs CICU/ICU), acute HF management timing, diagnostic algorithms, and (if relevant) PE workup.

Output JSON only:
{{"queries": ["...", "..."]}}
"""
    out = call_gpt_text(QUERYGEN_SYSTEM, prompt)
    data = json.loads(out)
    return data["queries"]

In [130]:
def retrieve_candidates_multiquery(case_text: str, queries: list[str], k_sections=25, k_recs=25) -> pd.DataFrame:
    all_hits = []
    for q in queries:
        a = faiss_search(sec_index, sec_meta, q, k_sections)   # your existing faiss_search
        b = faiss_search(rec_index, rec_meta, q, k_recs)
        a["source_index"] = "sections"
        b["source_index"] = "recommendations"
        a["query"] = q
        b["query"] = q
        all_hits.append(a)
        all_hits.append(b)

    cand = pd.concat(all_hits, ignore_index=True)

    # dedupe by chunk_id, keep best FAISS score
    cand = cand.sort_values("faiss_score", ascending=False).drop_duplicates("chunk_id").reset_index(drop=True)
    return cand

In [131]:
def build_context_bundle_tuned(case_text: str, mode: str, top_k_final: int = 14) -> dict:
    queries = generate_retrieval_queries_llm(case_text, mode=mode)

    candidates = retrieve_candidates_multiquery(case_text, queries, k_sections=30, k_recs=35)

    # rerank against the full case text (stronger than rerank vs query)
    top = rerank(case_text, candidates, top_k=top_k_final)  # your existing rerank()

    citations = [citation_object(top.iloc[i]) for i in range(len(top))]
    citations_pretty = [format_citation_row(top.iloc[i]) for i in range(len(top))]

    context_blocks = []
    for i in range(len(top)):
        cid = top.iloc[i]["chunk_id"]
        txt = top.iloc[i]["text"]
        context_blocks.append(f"[{cid}]\n{txt}")

    return {
        "mode": mode,
        "retrieval_queries": queries,
        "context_text": "\n\n---\n\n".join(context_blocks),
        "citations": citations,
        "citations_pretty": citations_pretty,
        "top_table": top
    }

In [132]:
bundleA = build_context_bundle_tuned(case_text, mode="A", top_k_final=14)
user_prompt_A = build_user_prompt(case_text, bundleA, task_instructions_A)

respA = call_gpt_json(SYSTEM_PROMPT_RAG, user_prompt_A)
respA

{'diagnoses_ranked': [{'diagnosis': 'Acute decompensated heart failure (HFrEF) with pulmonary congestion',
   'why': 'Breathlessness is a typical HF symptom and tachypnoea a less-specific HF sign, and CXR is a recommended test in suspected HF; this patient has known HFrEF and CXR shows cardiomegaly with bilateral interstitial markings consistent with congestion. [39f98d255431b22e]'},
  {'diagnosis': 'Acute pulmonary embolism (non-high-risk, hemodynamically stable)',
   'why': 'Acute dyspnoea with hypoxaemia and elevated D-dimer raises suspicion for PE, and PE risk stratification applies to patients without haemodynamic instability. [d9c4f53bd2f5fe14]'},
  {'diagnosis': 'Hyperkalaemia (acute, clinically significant)',
   'why': 'Clinical safety item (non-ESC).'}],
 'tests_ranked': [{'test': 'Transthoracic echocardiography (assess LV function and look for RV dysfunction if PE suspected)',
   'timing': 'Day 0 ER',
   'why': 'Echocardiography is recommended in suspected HF and is used to a

In [133]:
bundleB = build_context_bundle_tuned(case_text_post, mode="B", top_k_final=18)
user_prompt_B = build_user_prompt(case_text_post, bundleB, task_instructions_B)

respB = call_gpt_json(SYSTEM_PROMPT_RAG, user_prompt_B)
respB

{'q6_prognosis_score_1_to_8': 4,
 'q7_trigger': {'answer': 'Cannot be determined from provided data and retrieved guidelines.',
  'evidence': ['AHF may be precipitated by extrinsic factors, but specific triggers are not listed in provided context. [a3ab47a5293059fe]']},
 'q8_admit_to': {'answer': 'CARDIOLOGY DEPARTMENT',
  'why': 'AHF is severe enough to require urgent evaluation and commonly leads to unplanned hospital admission. [a3ab47a5293059fe] Echocardiography and ECG are recommended diagnostic tests during admission/hospitalization in AHF. [0b5c3c3ea6d8a18c]'},
 'q9_cause_workup': [{'step': 1,
   'do': 'ECG and serum troponin to assess for myocardial ischaemia/ACS as a potential precipitant',
   'when': 'Day 0 ER',
   'cite': ['0b5c3c3ea6d8a18c']},
  {'step': 2,
   'do': 'Measure natriuretic peptides (BNP/NT-proBNP) if diagnosis is uncertain and point-of-care assay available',
   'when': 'Day 0 ER',
   'cite': ['a3ab47a5293059fe']},
  {'step': 3,
   'do': 'Chest X-ray and/or lun

In [134]:
def render_bundle(bundle: dict, name: str = "bundle", top_n: int = 12, preview_chars: int = 900) -> str:
    lines = []
    lines.append(f"=== {name.upper()} — READABLE VIEW ===\n")

    # 1) retrieval queries
    qs = bundle.get("retrieval_queries", [])
    if qs:
        lines.append("Retrieval queries:")
        for i, q in enumerate(qs, 1):
            lines.append(f"  {i}. {q}")
        lines.append("")
    else:
        lines.append("Retrieval queries: (not present)\n")

    # 2) top_table
    top = bundle.get("top_table", None)
    if top is not None:
        lines.append(f"Top retrieved chunks (showing {min(top_n, len(top))}):")
        for i in range(min(top_n, len(top))):
            row = top.iloc[i]
            rr = row.get("rerank_score", None)
            src = row.get("source_index", "")
            title = str(row.get("doc_title", "")).strip()
            year = row.get("year", "")
            sect = str(row.get("section_title", "")).strip()
            page = row.get("page_start", "")
            cid = row.get("chunk_id", "")
            lines.append(f"  #{i+1} | rerank={rr:.3f} | {src} | {title} ({year}) p.{page}")
            lines.append(f"      section: {sect}")
            lines.append(f"      chunk_id: {cid}")
        lines.append("")
    else:
        lines.append("Top retrieved chunks: (top_table not present)\n")

    # 3) citations_pretty (audit list)
    cits = bundle.get("citations_pretty", [])
    if cits:
        lines.append("Citations (pretty):")
        for c in cits[:top_n]:
            lines.append(f"  - {c}")
        if len(cits) > top_n:
            lines.append(f"  ... ({len(cits)-top_n} more)")
        lines.append("")
    else:
        lines.append("Citations (pretty): (not present)\n")

    # 4) context preview
    ctx = bundle.get("context_text", "")
    lines.append(f"Context size: {len(ctx)} characters")
    if ctx:
        preview = ctx[:preview_chars].rstrip()
        lines.append("Context preview:")
        lines.append(preview + ("\n... (truncated)" if len(ctx) > preview_chars else ""))
    else:
        lines.append("Context preview: (empty)")

    return "\n".join(lines)

In [135]:
print(render_bundle(bundleA, name="bundleA (Query A)", top_n=12, preview_chars=900))

=== BUNDLEA (QUERY A) — READABLE VIEW ===

Retrieval queries:
  1. ESC acute heart failure guideline admission criteria emergency department discharge vs hospitalize
  2. ESC acute heart failure triage criteria ICU CICU vs cardiology ward hemodynamically stable
  3. ESC acute heart failure initial assessment algorithm emergency department dyspnea pulmonary congestion
  4. ESC acute heart failure diagnostic workup natriuretic peptides BNP NT-proBNP chest x-ray echocardiography timing
  5. ESC acute heart failure management first hours IV diuretics vasodilators oxygen noninvasive ventilation indications
  6. ESC acute heart failure risk stratification predictors of early mortality need for intensive care
  7. ESC guideline acute heart failure hypertensive acute heart failure management vasodilators blood pressure 160/110
  8. ESC guideline acute heart failure troponin interpretation myocardial infarction rule-in rule-out LBBB
  9. ESC pulmonary embolism guideline diagnostic algorithm hem

In [136]:
PROGNOSIS_SCALE_1_8 = {
    1: "Safe for Discharge from ER",
    2: "Short Observation (<24h)",
    3: "Short Admission (1–3 days)",
    4: "Standard Admission (3–7 days)",
    5: "Prolonged Admission (>7 days)",
    6: "High Risk of ICU",
    7: "High Risk of Intubation",
    8: "High Risk of Death (In-hospital)"
}

def render_query_a_answers(resp: dict) -> str:
    lines = []
    lines.append("=== QUERY A (Pre-diagnosis): Readable Answers ===\n")

    # 1) Diagnoses
    lines.append("1) Differential diagnoses (ranked)")
    dxs = resp.get("diagnoses_ranked", [])
    for i, d in enumerate(dxs, 1):
        lines.append(f"  {i}. {d.get('diagnosis','')}")
        lines.append(f"     Why: {d.get('why','')}")
    lines.append("")

    # 2) Tests
    lines.append("2) Further tests (ranked) + timing")
    tests = resp.get("tests_ranked", [])
    for i, t in enumerate(tests, 1):
        lines.append(f"  {i}. {t.get('test','')}")
        lines.append(f"     Timing: {t.get('timing','')}")
        lines.append(f"     Why: {t.get('why','')}")
    lines.append("")

    # 3) Disposition
    disp = resp.get("disposition", {})
    lines.append("3) Disposition")
    lines.append(f"  Answer: {disp.get('answer','')}")
    lines.append(f"  Why: {disp.get('why','')}")
    lines.append("")

    # 4) Admit to
    adm = resp.get("admit_to", {})
    lines.append("4) Admission destination")
    # sometimes you included 'answer_rule' in the object; show it only if present
    if "answer_rule" in adm:
        lines.append(f"  Rule: {adm.get('answer_rule')}")
    lines.append(f"  Answer: {adm.get('answer','')}")
    lines.append(f"  Why: {adm.get('why','')}")
    lines.append("")

    # 5) Prognosis per diagnosis
    lines.append("5) Prognosis per diagnosis (score 1–8)")
    progs = resp.get("prognosis_per_diagnosis", [])
    for p in progs:
        dx = p.get("diagnosis","")
        sc = p.get("score_1_to_8", None)
        label = PROGNOSIS_SCALE_1_8.get(sc, "Unknown") if isinstance(sc, int) else "Unknown"
        why = p.get("why","")
        lines.append(f"  - {dx}: {sc} — {label}")
        lines.append(f"    Why: {why}")

    return "\n".join(lines)

In [137]:
HDS_SCALE_1_5 = {
    1: "Discharged from ER (<24h)",
    2: "Short Stay (1–2 Days)",
    3: "Standard Stay (3–5 Days)",
    4: "Prolonged Stay (6–10 Days)",
    5: "Extended Stay (>10 Days)"
}

def render_query_b_answers(resp: dict) -> str:
    lines = []
    lines.append("=== QUERY B (Post-diagnosis): Readable Answers ===\n")

    # Q6
    q6 = resp.get("q6_prognosis_score_1_to_8", None)
    lines.append(f"6) Prognosis score (1–8): {q6}")
    lines.append("")

    # Q7
    q7 = resp.get("q7_trigger", {})
    lines.append("7) Most likely trigger of decompensation")
    lines.append(f"  Answer: {q7.get('answer','')}")
    evidence = q7.get("evidence", [])
    if evidence:
        lines.append("  Evidence:")
        for e in evidence:
            lines.append(f"    • {e}")
    lines.append("")

    # Q8
    q8 = resp.get("q8_admit_to", {})
    lines.append("8) Admit to")
    lines.append(f"  Answer: {q8.get('answer','')}")
    lines.append(f"  Why: {q8.get('why','')}")
    lines.append("")

    # Q9
    q9 = resp.get("q9_cause_workup", [])
    lines.append("9) Workup to identify the cause (algorithm)")
    for step in q9:
        lines.append(f"  Step {step.get('step')}: {step.get('do','')}")
        lines.append(f"     When: {step.get('when','')}")
        lines.append(f"     Cite: {', '.join(step.get('cite', []))}")
    lines.append("")

    # Q10
    q10 = resp.get("q10_treatment_plan", [])
    lines.append("10) Treatment plan")
    for dayblock in q10:
        lines.append(f"  {dayblock.get('day')}:")
        for a in dayblock.get("do", []):
            lines.append(f"    • {a}")
        cits = dayblock.get("cite", [])
        if cits:
            lines.append(f"    Cite: {', '.join(cits)}")
    lines.append("")

    # Q11
    q11 = resp.get("q11_followup", {})
    lines.append("11) Follow-up investigations / interventions")
    lines.append("  In-hospital:")
    for item in q11.get("in_hospital", []):
        lines.append(f"    • {item}")
    lines.append("  After discharge:")
    for item in q11.get("after_discharge", []):
        lines.append(f"    • {item}")
    lines.append("")

    # Q12
    q12 = resp.get("q12_hds_score_1_to_5", None)
    label = HDS_SCALE_1_5.get(q12, "Unknown") if isinstance(q12, int) else "Unknown"
    lines.append(f"12) Estimated duration (HDS-5): {q12} — {label}")

    return "\n".join(lines)

In [138]:
print(render_query_a_answers(respA))

=== QUERY A (Pre-diagnosis): Readable Answers ===

1) Differential diagnoses (ranked)
  1. Acute decompensated heart failure (HFrEF) with pulmonary congestion
     Why: Breathlessness is a typical HF symptom and tachypnoea a less-specific HF sign, and CXR is a recommended test in suspected HF; this patient has known HFrEF and CXR shows cardiomegaly with bilateral interstitial markings consistent with congestion. [39f98d255431b22e]
  2. Acute pulmonary embolism (non-high-risk, hemodynamically stable)
     Why: Acute dyspnoea with hypoxaemia and elevated D-dimer raises suspicion for PE, and PE risk stratification applies to patients without haemodynamic instability. [d9c4f53bd2f5fe14]
  3. Hyperkalaemia (acute, clinically significant)
     Why: Clinical safety item (non-ESC).

2) Further tests (ranked) + timing
  1. Transthoracic echocardiography (assess LV function and look for RV dysfunction if PE suspected)
     Timing: Day 0 ER
     Why: Echocardiography is recommended in suspected H

In [139]:
print(render_query_b_answers(respB))

=== QUERY B (Post-diagnosis): Readable Answers ===

6) Prognosis score (1–8): 4

7) Most likely trigger of decompensation
  Answer: Cannot be determined from provided data and retrieved guidelines.
  Evidence:
    • AHF may be precipitated by extrinsic factors, but specific triggers are not listed in provided context. [a3ab47a5293059fe]

8) Admit to
  Answer: CARDIOLOGY DEPARTMENT
  Why: AHF is severe enough to require urgent evaluation and commonly leads to unplanned hospital admission. [a3ab47a5293059fe] Echocardiography and ECG are recommended diagnostic tests during admission/hospitalization in AHF. [0b5c3c3ea6d8a18c]

9) Workup to identify the cause (algorithm)
  Step 1: ECG and serum troponin to assess for myocardial ischaemia/ACS as a potential precipitant
     When: Day 0 ER
     Cite: 0b5c3c3ea6d8a18c
  Step 2: Measure natriuretic peptides (BNP/NT-proBNP) if diagnosis is uncertain and point-of-care assay available
     When: Day 0 ER
     Cite: a3ab47a5293059fe
  Step 3: Chest

Version 4.0

In [140]:
from openai import OpenAI
import json, re

client = OpenAI()
MODEL_NAME = "gpt-5.2"  # keep as your target name

def call_gpt(system_prompt: str, user_prompt: str, temperature: float = 0.0) -> str:
    resp = client.responses.create(
        model=MODEL_NAME,
        temperature=temperature,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return resp.output_text

def extract_json(text: str) -> dict:
    # robust JSON extraction: find first {...} block
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        raise ValueError("No JSON object found in model output.")
    return json.loads(m.group(0))

In [141]:
def parse_case_snapshot(case_text: str) -> dict:
    t = case_text.lower()

    def find_bp():
        m = re.search(r"blood pressure\s*=\s*(\d+)\s*/\s*(\d+)", t)
        return {"sys": int(m.group(1)), "dia": int(m.group(2))} if m else None

    def find_spo2():
        m = re.search(r"spo2\s*=\s*(\d+)", t)
        return int(m.group(1)) if m else None

    def find_po2():
        m = re.search(r"po₂\s*=\s*(\d+)|po2\s*=\s*(\d+)", t)
        if not m: return None
        return int(next(g for g in m.groups() if g))

    def find_k():
        m = re.search(r"potassium\s*=\s*([0-9.]+)", t)
        return float(m.group(1)) if m else None

    snap = {
        "bp": find_bp(),
        "spo2": find_spo2(),
        "po2": find_po2(),
        "k": find_k(),
        "known_hf": "heart failure" in t and ("hfr" in t or "reduced ejection fraction" in t),
        "cad": any(x in t for x in ["nstemi","cabg","pci","coronary"]),
        "c ინr": "lbbb" in t,
        "d_dimer_present": "d-dimer" in t or "d-dimers" in t,
        "red_flags": []
    }

    # simple red flags
    if snap["k"] is not None and snap["k"] >= 6.0:
        snap["red_flags"].append("hyperkalemia>=6.0")
    if snap["spo2"] is not None and snap["spo2"] < 90:
        snap["red_flags"].append("hypoxemia_spo2<90")
    if snap["bp"] and snap["bp"]["sys"] < 90:
        snap["red_flags"].append("hypotension_sys<90")

    return snap

In [142]:
QUERYGEN_SYSTEM = """
You generate short retrieval queries for ESC guideline passages.
Output only JSON: {"queries":[...]}.
"""

def generate_queries(case_text: str, question_set: str) -> list[str]:
    prompt = f"""
Case summary:
{case_text}

Question set: {question_set}

Generate 10 retrieval queries that will pull guideline text to answer:
- differential diagnosis for dyspnea in cardiac patients
- admission vs discharge criteria and monitoring level (ward vs CICU/ICU)
- acute heart failure initial evaluation and treatment steps
- pulmonary embolism diagnostic + risk stratification pathway (if relevant)
- key tests and their timing in ED vs inpatient vs discharge

Rules:
- Queries must be short (8–16 words).
- Use guideline-like wording: "recommended", "should", "initial assessment", "risk stratification", "admission", "discharge", "monitoring".
Return JSON only.
"""
    raw = call_gpt(QUERYGEN_SYSTEM, prompt, temperature=0.0)
    data = extract_json(raw)
    return data["queries"]

In [143]:
def build_context_bundle_from_queries(queries: list[str], top_k_final: int = 14) -> dict:
    # You already have: faiss_search(), rerank(), citation_object(), format_citation_row()
    # This function assumes those exist in your notebook.

    all_hits = []
    for q in queries:
        a = faiss_search(sec_index, sec_meta, q, k=30)
        b = faiss_search(rec_index, rec_meta, q, k=35)
        a["source_index"] = "sections"
        b["source_index"] = "recommendations"
        a["query"] = q
        b["query"] = q
        all_hits.append(a); all_hits.append(b)

    cand = pd.concat(all_hits, ignore_index=True)
    cand = cand.sort_values("faiss_score", ascending=False).drop_duplicates("chunk_id").reset_index(drop=True)

    top = rerank(" ".join(queries), cand, top_k=top_k_final)  # rerank to query-intent not single patient

    context_blocks = [f"[{row.chunk_id}]\n{row.text}" for row in top.itertuples(index=False)]

    return {
        "retrieval_queries": queries,
        "top_table": top,
        "context_text": "\n\n---\n\n".join(context_blocks),
        "citations_pretty": [format_citation_row(top.iloc[i]) for i in range(len(top))]
    }

In [144]:
SYSTEM_PROMPT_ALWAYS_ANSWER = """
You are evaluating clinical reasoning for research only (NOT real patient care).

Hard rules:
1) You MUST ALWAYS provide an answer for every question (no refusal like "cannot be determined").
2) Separate two layers:
   - Guideline support: only what is directly supported by the provided context excerpts, with citations [chunk_id].
   - Clinical judgment: your best medical reasoning even if guideline support is missing.
3) If there is no direct supporting excerpt, write: "No direct guideline citation retrieved for this point."
4) Keep it readable for clinicians: short bullets, no essays.
5) If there are safety-critical abnormalities (e.g., K>=6.0), include them under safety_critical.

You MUST output a single JSON object and nothing else.
"""

In [145]:
def build_prompt_query_A(case_text: str, bundle: dict) -> str:
    snap = parse_case_snapshot(case_text)

    return f"""
CASE_TEXT:
{case_text}

CASE_SNAPSHOT (auto-extracted):
{json.dumps(snap, indent=2)}

RETRIEVED_GUIDELINE_EXCERPTS:
{bundle["context_text"]}

TASK:
Answer Question Set A in JSON with this exact structure:

{{
  "differential": [
    {{"dx": "string", "probability": 0.0, "one_liner": "string", "evidence": [{{"chunk_id":"..."}}], "confidence_0_to_1": 0.0}}
  ],
  "tests": [
    {{"test":"string","timing":"ER Day0 | Inpatient Day1 | Discharge/Outpatient","reason":"string","evidence":[{{"chunk_id":"..."}}]}}
  ],
  "disposition": {{
    "answer":"ADMIT | DISCHARGE | OBSERVE",
    "department":"CARDIOLOGY DEPARTMENT | CICU | RESPIRATORY DEPARTMENT (PULMONOLOGISTS) | INTERNAL MEDICINE | ICU | SURGICAL",
    "confidence_0_to_1": 0.0,
    "guideline_support":"string with citations or 'No direct guideline citation retrieved for this point.'",
    "clinical_judgment":"string (doctor style, 1–2 sentences)"
  }},
  "prognosis": [
    {{"dx":"string","score_1_to_8":1,"confidence_0_to_1":0.0,"justification":"string","evidence":[{{"chunk_id":"..."}}]}}
  ],
  "safety_critical": [
    {{"item":"string","action":"string","why":"string","evidence":[{{"chunk_id":"..."}}],"severity":"HIGH|MED|LOW"}}
  ]
}}

Constraints:
- differential: exactly 3 items; probabilities must sum to 1.0
- tests: max 7 items
- prognosis: exactly 3 items corresponding to differential
- Always fill disposition with a concrete answer and department.
- Evidence can be empty, but then guideline_support MUST say the no-citation sentence.
"""

In [146]:
queriesA = generate_queries(case_text, question_set="A")
bundleA = build_context_bundle_from_queries(queriesA, top_k_final=16)

promptA = build_prompt_query_A(case_text, bundleA)
rawA = call_gpt(SYSTEM_PROMPT_ALWAYS_ANSWER, promptA, temperature=0.0)
respA = extract_json(rawA)
respA

{'differential': [{'dx': 'Acute decompensated heart failure / acute pulmonary oedema (hypertensive AHF phenotype) in known HFrEF',
   'probability': 0.6,
   'one_liner': 'Known HFrEF with acute dyspnea, hypoxemia (PaO2 65 on room air), tachypnea, cardiomegaly and bilateral interstitial markings on CXR, with markedly elevated BP suggesting hypertensive AHF/pulmonary oedema physiology.',
   'evidence': [{'chunk_id': '0b5c3c3ea6d8a18c'},
    {'chunk_id': 'ad4f02e2c45bdb81'}],
   'confidence_0_to_1': 0.72},
  {'dx': 'Acute pulmonary embolism (non-high-risk PE) vs PE as precipitant of dyspnea',
   'probability': 0.25,
   'one_liner': 'Acute dyspnea with hypoxemia and elevated D-dimer; hemodynamically stable, so if PE present would likely be non-high-risk requiring risk stratification (PESI/sPESI) and RV assessment.',
   'evidence': [{'chunk_id': 'ad4f02e2c45bdb81'},
    {'chunk_id': '1c9ae0a89fb33aab'},
    {'chunk_id': 'a8442d402e5e8b51'}],
   'confidence_0_to_1': 0.5},
  {'dx': 'Acute cor

In [147]:
def pretty_query_A(resp: dict) -> str:
    lines = []
    lines.append("=== QUERY A REPORT ===\n")

    lines.append("1) Differential (ranked)")
    for i, d in enumerate(resp["differential"], 1):
        lines.append(f"  {i}. {d['dx']}  (p={d['probability']:.2f}, conf={d['confidence_0_to_1']:.2f})")
        lines.append(f"     {d['one_liner']}")
        ev = [e.get("chunk_id") for e in d.get("evidence", []) if e.get("chunk_id")]
        if ev:
            lines.append(f"     Evidence: {', '.join(ev)}")
    lines.append("")

    lines.append("2) Tests + timing")
    for i, t in enumerate(resp["tests"], 1):
        lines.append(f"  {i}. {t['test']} — {t['timing']}")
        lines.append(f"     Reason: {t['reason']}")
        ev = [e.get("chunk_id") for e in t.get("evidence", []) if e.get("chunk_id")]
        if ev:
            lines.append(f"     Evidence: {', '.join(ev)}")
    lines.append("")

    disp = resp["disposition"]
    lines.append("3) Disposition")
    lines.append(f"  Decision: {disp['answer']} → {disp['department']}  (conf={disp['confidence_0_to_1']:.2f})")
    lines.append(f"  Guideline support: {disp['guideline_support']}")
    lines.append(f"  Clinical judgment: {disp['clinical_judgment']}")
    lines.append("")

    lines.append("4) Prognosis (1–8)")
    for p in resp["prognosis"]:
        lines.append(f"  - {p['dx']}: {p['score_1_to_8']} (conf={p['confidence_0_to_1']:.2f})")
        lines.append(f"    {p['justification']}")
    lines.append("")

    lines.append("5) Safety-critical")
    if resp["safety_critical"]:
        for s in resp["safety_critical"]:
            lines.append(f"  - [{s['severity']}] {s['item']}")
            lines.append(f"    Action: {s['action']}")
            lines.append(f"    Why: {s['why']}")
    else:
        lines.append("  (none)")

    return "\n".join(lines)

print(pretty_query_A(respA))

=== QUERY A REPORT ===

1) Differential (ranked)
  1. Acute decompensated heart failure / acute pulmonary oedema (hypertensive AHF phenotype) in known HFrEF  (p=0.60, conf=0.72)
     Known HFrEF with acute dyspnea, hypoxemia (PaO2 65 on room air), tachypnea, cardiomegaly and bilateral interstitial markings on CXR, with markedly elevated BP suggesting hypertensive AHF/pulmonary oedema physiology.
     Evidence: 0b5c3c3ea6d8a18c, ad4f02e2c45bdb81
  2. Acute pulmonary embolism (non-high-risk PE) vs PE as precipitant of dyspnea  (p=0.25, conf=0.50)
     Acute dyspnea with hypoxemia and elevated D-dimer; hemodynamically stable, so if PE present would likely be non-high-risk requiring risk stratification (PESI/sPESI) and RV assessment.
     Evidence: ad4f02e2c45bdb81, 1c9ae0a89fb33aab, a8442d402e5e8b51
  3. Acute coronary syndrome / myocardial ischemia as trigger of AHF (NSTEMI/unstable angina)  (p=0.15, conf=0.42)
     Extensive CAD history and new/worsening dyspnea can be an anginal equiva

In [148]:
def build_prompt_query_B(case_text_post: str, bundle: dict) -> str:
    snap = parse_case_snapshot(case_text_post)

    return f"""
CASE_TEXT (post-diagnosis):
{case_text_post}

CASE_SNAPSHOT (auto-extracted):
{json.dumps(snap, indent=2)}

RETRIEVED_GUIDELINE_EXCERPTS:
{bundle["context_text"]}

TASK:
Answer Question Set B (after diagnosis) in JSON with this exact structure:

{{
  "prognosis": {{
    "score_1_to_8": 1,
    "confidence_0_to_1": 0.0,
    "guideline_support": "short text + citations OR 'No direct guideline citation retrieved for this point.'",
    "clinical_judgment": "1–2 sentences"
  }},
  "most_likely_trigger": {{
    "answer": "string",
    "confidence_0_to_1": 0.0,
    "guideline_support": "short text + citations OR 'No direct guideline citation retrieved for this point.'",
    "clinical_judgment": "1–2 sentences"
  }},
  "admission_level": {{
    "department": "CICU | CARDIOLOGY DEPARTMENT",
    "confidence_0_to_1": 0.0,
    "guideline_support": "short text + citations OR 'No direct guideline citation retrieved for this point.'",
    "clinical_judgment": "1–2 sentences"
  }},
  "cause_workup_algorithm": [
    {{
      "step": 1,
      "action": "string",
      "timing": "Day 0 | Day 1 | Day 2 | Discharge/Outpatient",
      "purpose": "string",
      "guideline_support": "short text + citations OR 'No direct guideline citation retrieved for this point.'",
      "clinical_judgment": "≤1 sentence"
    }}
  ],
  "treatment_plan_by_day": [
    {{
      "day": "Day 0",
      "actions": ["string", "string"],
      "guideline_support": "short text + citations OR 'No direct guideline citation retrieved for this point.'",
      "clinical_judgment": "≤2 sentences"
    }},
    {{
      "day": "Day 1",
      "actions": ["string"],
      "guideline_support": "short text + citations OR 'No direct guideline citation retrieved for this point.'",
      "clinical_judgment": "≤2 sentences"
    }},
    {{
      "day": "Day 2",
      "actions": ["string"],
      "guideline_support": "short text + citations OR 'No direct guideline citation retrieved for this point.'",
      "clinical_judgment": "≤2 sentences"
    }},
    {{
      "day": "Discharge",
      "actions": ["string"],
      "guideline_support": "short text + citations OR 'No direct guideline citation retrieved for this point.'",
      "clinical_judgment": "≤2 sentences"
    }}
  ],
  "followup": {{
    "in_hospital": [
      {{
        "item": "string",
        "purpose": "string",
        "guideline_support": "short text + citations OR 'No direct guideline citation retrieved for this point.'",
        "clinical_judgment": "≤1 sentence"
      }}
    ],
    "after_discharge": [
      {{
        "item": "string",
        "purpose": "string",
        "guideline_support": "short text + citations OR 'No direct guideline citation retrieved for this point.'",
        "clinical_judgment": "≤1 sentence"
      }}
    ]
  }},
  "estimated_duration": {{
    "hds_score_1_to_5": 1,
    "confidence_0_to_1": 0.0,
    "guideline_support": "short text + citations OR 'No direct guideline citation retrieved for this point.'",
    "clinical_judgment": "1–2 sentences"
  }},
  "safety_critical": [
    {{
      "item": "string",
      "action": "string",
      "why": "string",
      "severity": "HIGH|MED|LOW",
      "guideline_support": "short text + citations OR 'No direct guideline citation retrieved for this point.'",
      "clinical_judgment": "≤1 sentence"
    }}
  ]
}}

Constraints:
- Always answer every field (no 'cannot be determined').
- Keep actions short and practical.
- cause_workup_algorithm: 6–10 steps.
- treatment_plan_by_day: keep Day 0/1/2/Discharge, max 6 bullets per day.
- followup: max 6 items in_hospital, max 6 items after_discharge.
- If you mention anticoagulation/diuresis/vasodilators/oxygen/NIV etc, do so as general actions consistent with the case (no dosing).
Return ONLY JSON.
"""

In [149]:
queriesB = generate_queries(case_text_post, question_set="B")
bundleB = build_context_bundle_from_queries(queriesB, top_k_final=18)

promptB = build_prompt_query_B(case_text_post, bundleB)
rawB = call_gpt(SYSTEM_PROMPT_ALWAYS_ANSWER, promptB, temperature=0.0)
respB = extract_json(rawB)

respB

{'prognosis': {'score_1_to_8': 4,
  'confidence_0_to_1': 0.62,
  'guideline_support': 'AHF is associated with meaningful in-hospital mortality (4–10%) and high post-discharge event rates, supporting need for inpatient management rather than ER discharge. [0c7ef2e8ac2c1598]',
  'clinical_judgment': 'Hemodynamically stable but hypoxemic (PaO2 65), hypertensive, and with significant hyperkalemia suggests moderate severity requiring stabilization and monitoring; not an ICU-level shock picture.'},
 'most_likely_trigger': {'answer': 'Hypertensive acute decompensated HFrEF (afterload-driven pulmonary congestion/redistribution), possibly with medication/diet nonadherence or volume/salt excess',
  'confidence_0_to_1': 0.55,
  'guideline_support': 'Guidelines emphasize identifying precipitants in AHF and list hypertension emergency among acute aetiologies to identify and treat early. [15b272d975dcd452]',
  'clinical_judgment': 'Marked BP elevation with interstitial edema on CXR and acute dyspnea

In [150]:
HDS_SCALE_1_5 = {
    1: "Discharged from ER (<24h)",
    2: "Short Stay (1–2 days)",
    3: "Standard Stay (3–5 days)",
    4: "Prolonged Stay (6–10 days)",
    5: "Extended Stay (>10 days)"
}

def pretty_query_B(resp: dict) -> str:
    lines = []
    lines.append("=== QUERY B REPORT (Post-diagnosis) ===\n")

    # 6 prognosis
    prog = resp["prognosis"]
    lines.append("6) Prognosis")
    lines.append(f"  Score (1–8): {prog['score_1_to_8']}  (conf={prog['confidence_0_to_1']:.2f})")
    lines.append(f"  Guideline support: {prog['guideline_support']}")
    lines.append(f"  Clinical judgment: {prog['clinical_judgment']}")
    lines.append("")

    # 7 trigger
    trg = resp["most_likely_trigger"]
    lines.append("7) Most likely trigger of decompensation")
    lines.append(f"  Answer: {trg['answer']}  (conf={trg['confidence_0_to_1']:.2f})")
    lines.append(f"  Guideline support: {trg['guideline_support']}")
    lines.append(f"  Clinical judgment: {trg['clinical_judgment']}")
    lines.append("")

    # 8 admit level
    adm = resp["admission_level"]
    lines.append("8) Admit to")
    lines.append(f"  Department: {adm['department']}  (conf={adm['confidence_0_to_1']:.2f})")
    lines.append(f"  Guideline support: {adm['guideline_support']}")
    lines.append(f"  Clinical judgment: {adm['clinical_judgment']}")
    lines.append("")

    # 9 workup algorithm
    lines.append("9) Workup to identify the cause (algorithm)")
    for s in resp["cause_workup_algorithm"]:
        lines.append(f"  Step {s['step']}: {s['action']} — {s['timing']}")
        lines.append(f"     Purpose: {s['purpose']}")
        lines.append(f"     Guideline support: {s['guideline_support']}")
        lines.append(f"     Clinical judgment: {s['clinical_judgment']}")
    lines.append("")

    # 10 treatment by day
    lines.append("10) Treatment plan by day")
    for dayblock in resp["treatment_plan_by_day"]:
        lines.append(f"  {dayblock['day']}:")
        for a in dayblock["actions"]:
            lines.append(f"    • {a}")
        lines.append(f"    Guideline support: {dayblock['guideline_support']}")
        lines.append(f"    Clinical judgment: {dayblock['clinical_judgment']}")
    lines.append("")

    # 11 followup
    lines.append("11) Follow-up")
    lines.append("  In-hospital:")
    for item in resp["followup"]["in_hospital"]:
        lines.append(f"    • {item['item']} — {item['purpose']}")
        lines.append(f"      Guideline support: {item['guideline_support']}")
        lines.append(f"      Clinical judgment: {item['clinical_judgment']}")
    lines.append("  After discharge:")
    for item in resp["followup"]["after_discharge"]:
        lines.append(f"    • {item['item']} — {item['purpose']}")
        lines.append(f"      Guideline support: {item['guideline_support']}")
        lines.append(f"      Clinical judgment: {item['clinical_judgment']}")
    lines.append("")

    # 12 duration
    dur = resp["estimated_duration"]
    label = HDS_SCALE_1_5.get(dur["hds_score_1_to_5"], "Unknown")
    lines.append("12) Estimated duration of hospitalization")
    lines.append(f"  HDS-5: {dur['hds_score_1_to_5']} — {label}  (conf={dur['confidence_0_to_1']:.2f})")
    lines.append(f"  Guideline support: {dur['guideline_support']}")
    lines.append(f"  Clinical judgment: {dur['clinical_judgment']}")
    lines.append("")

    # safety
    lines.append("Safety-critical")
    if resp["safety_critical"]:
        for s in resp["safety_critical"]:
            lines.append(f"  - [{s['severity']}] {s['item']}")
            lines.append(f"    Action: {s['action']}")
            lines.append(f"    Why: {s['why']}")
            lines.append(f"    Guideline support: {s['guideline_support']}")
            lines.append(f"    Clinical judgment: {s['clinical_judgment']}")
    else:
        lines.append("  (none)")

    return "\n".join(lines)

print(pretty_query_B(respB))

=== QUERY B REPORT (Post-diagnosis) ===

6) Prognosis
  Score (1–8): 4  (conf=0.62)
  Guideline support: AHF is associated with meaningful in-hospital mortality (4–10%) and high post-discharge event rates, supporting need for inpatient management rather than ER discharge. [0c7ef2e8ac2c1598]
  Clinical judgment: Hemodynamically stable but hypoxemic (PaO2 65), hypertensive, and with significant hyperkalemia suggests moderate severity requiring stabilization and monitoring; not an ICU-level shock picture.

7) Most likely trigger of decompensation
  Answer: Hypertensive acute decompensated HFrEF (afterload-driven pulmonary congestion/redistribution), possibly with medication/diet nonadherence or volume/salt excess  (conf=0.55)
  Guideline support: Guidelines emphasize identifying precipitants in AHF and list hypertension emergency among acute aetiologies to identify and treat early. [15b272d975dcd452]
  Clinical judgment: Marked BP elevation with interstitial edema on CXR and acute dyspnea